# Extraccion de caracteristicas

In [13]:
# Cargar los datos transformados
import os
from os import path
import numpy as np
from pathlib import Path

STFT_PATH = './data/tf_representations/stft.npz'
CWT_PATH  = './data/tf_representations/cwt.npz'
SST_PATH  = './data/tf_representations/sst.npz'
WPD_PATH  = './data/tf_representations/wpd.npz'
#GSP_PATH  = './data/tf_representations/gsp.npz'

CHANNELS   = ['AF3','F7','F3','FC5','T7','P7','O1','O2','P8','T8','FC6','F4','F8','AF4']
TARGET_COL = 'eyeDetection'  # 0: ojos abiertos, 1: ojos cerrados

#lista = [('STFT', STFT_PATH), ('CWT', CWT_PATH), ('SST', SST_PATH), ('WPD', WPD_PATH), ('GSP', GSP_PATH)]
lista = [('STFT', STFT_PATH), ('CWT', CWT_PATH), ('SST', SST_PATH), ('WPD', WPD_PATH)]


for label, fpath in lista:
    if not path.exists(fpath):
        raise FileNotFoundError(f"El archivo no existe: {fpath}")
    else:
        print(f"Los datos {label} ya existen en {fpath}")

stft = np.load(STFT_PATH)
cwt  = np.load(CWT_PATH)
sst  = np.load(SST_PATH)
wpd  = np.load(WPD_PATH)
#gsp  = np.load(GSP_PATH, allow_pickle=True)

print(f"\nSTFT — Shape por canal : {stft['O2'].shape}  |  Canales: {len(CHANNELS)}")
print(f"CWT  — Shape por canal : {cwt['O2'].shape}")
print(f"SST  — Shape por canal : {sst['O2'].shape}")
print(f"WPD  — Shape por canal : {wpd['O2'].shape}")
#print(f"GSP  — Shape espectrograma: {gsp['spectrogram'].shape}  (modos × tiempo)")

print(f"\nLabels STFT: {stft['labels'].shape}  |  Clases: {np.unique(stft['labels'])}")
print(f"Labels CWT : {cwt['labels'].shape}")
print(f"Labels SST : {sst['labels'].shape}")
print(f"Labels WPD : {wpd['labels'].shape}")
#print(f"Labels GSP : {gsp['labels'].shape}   |  Clases: {np.unique(gsp['labels'])}")

Los datos STFT ya existen en ./data/tf_representations/stft.npz
Los datos CWT ya existen en ./data/tf_representations/cwt.npz
Los datos SST ya existen en ./data/tf_representations/sst.npz
Los datos WPD ya existen en ./data/tf_representations/wpd.npz

STFT — Shape por canal : (73, 236)  |  Canales: 14
CWT  — Shape por canal : (80, 14980)
SST  — Shape por canal : (80, 14980)
WPD  — Shape por canal : (9, 231)

Labels STFT: (236,)  |  Clases: [0. 1.]
Labels CWT : (14980,)
Labels SST : (14980,)
Labels WPD : (231,)


## Consideraciones y definiciones iniciales
- **Alineación temporal**: 
    - **STFT y CWT**: La STFT ya tiene sus propias ventanas de 2s (~232 ventanas en stft_times). Para que la comparación sea justa, la CWT debe analizar exactamente los mismos segmentos temporales. Para cada ventana STFT centrada en t_i, calculo los índices de muestra correspondientes en la CWT y promedio la potencia en ese eje temporal, obteniendo un espectro promedio (80,). Ambas representaciones producen así el mismo número de instancias con las mismas etiquetas.
    - **SST:** comparte exactamente la misma estructura que la CWT — una columna por muestra (resolución temporal completa, 14980 puntos), con frecuencias log-espaciadas en 4–40 Hz. Por lo tanto, se alinea con las ventanas STFT de la misma forma que la CWT: promediando la potencia sobre el rango de muestras correspondiente a cada ventana.
    - **WPD:** a diferencia de SST/CWT, la WPD ya fue calculada con su propio ventaneo deslizante (mismo `window_sec=2` y `overlap_pct=0.75` que la STFT), en el notebook anterior. Esto significa que **no hay que volver a ventanear** — cada columna de `wpd[ch]` ya es la potencia promedio de una ventana de 2s. 
    
        Sin embargo, el ventaneo de la WPD se calculó de forma independiente al de `scipy.signal.stft`, con un manejo de bordes distinto (relleno de ceros vs. recorte estricto). Esto produce un número de ventanas y tiempos centrales ligeramente distintos entre ambas técnicas. Para alinear ambas grillas, se busca para cada ventana STFT la ventana WPD con el tiempo central más cercano, y se verifica que ese desfase sea despreciable frente al largo de ventana (2 s).

- **Integración trapezoidal**: Las frecuencias de la STFT son linealmente espaciadas y las de la CWT son log-espaciadas. Si sumamos directamente los valores de potencia sin ponderar vamos a sobrepasar las frecuencias bajas en la CWT. `np.trapezoid(power, freqs)` maneja ambos casos correctamente.
- **Origen de frecuencias CWT**: Según como se construyeron las escalas en el notebook anterior, las frecuencias de la CWT pueden estar en orden descendente. Hay una función auxiliar que lo detecta y corrige antes de integrar.

## Decisiones de diseño — extracción de features

### Features seleccionadas (4 por canal × 14 canales = 56 features)

Para cada ventana temporal y cada canal EEG se extraen cuatro características
desde el espectro de potencia promediado:

1. **Potencia alfa absoluta** (`alpha_abs`): integral trapezoidal de la densidad
   de potencia en 8–13 Hz. Cuantifica la energía bruta del ritmo alfa. Se usa
   integración trapezoidal (`np.trapz`) en lugar de suma directa para manejar
   correctamente frecuencias no uniformemente espaciadas (CWT log-espaciada).

2. **Potencia alfa relativa** (`alpha_rel`): cociente entre la potencia alfa y la
   potencia total en 4–40 Hz. Al normalizar por la potencia total, esta feature
   es invariante a diferencias de amplitud entre sujetos y sesiones — un requisito
   importante para comparaciones justas entre técnicas TF con distintas unidades.

3. **Entropía espectral de Shannon normalizada** (`entropy`): mide cuán
   concentrada está la energía en el espectro completo (4–40 Hz). Un valor bajo
   indica que la energía se concentra en pocas frecuencias (ej. alfa dominante);
   un valor alto indica distribución uniforme (ausencia de ritmo dominante).
   Se normaliza por `log2(N)` para que el rango sea siempre [0, 1]
   independientemente del número de bins de frecuencia.

4. **Centro de gravedad alfa** (`cog`): media ponderada de las frecuencias en
   8–13 Hz, con la potencia como peso — método de Klimesch (1999). Más robusto
   que el pico máximo porque no asume que el alfa tiene un único pico, lo cual
   no siempre se cumple empíricamente.

### Rango de frecuencias

Ambas representaciones cubren 4–40 Hz, alineado con el filtro pasa-banda
aplicado en el preprocesamiento. La banda delta (< 4 Hz) se excluye porque
no es relevante para la tarea de ojos abiertos/cerrados.

In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --------------------------------------------------------------
# ------------------------- CONSTANTES -------------------------
# --------------------------------------------------------------

FS       = 128    # Hz
ALPHA_LO = 8.0    # Hz — límite inferior banda alfa
ALPHA_HI = 13.0   # Hz — límite superior banda alfa
WIN_SEC  = 2.0    # s  — largo de ventana (idéntico al de la STFT)

### Funciones de Extracción

In [15]:
def _sort_asc(power, freqs):
    """Reordena ascendentemente si las frecuencias vienen descendentes (CWT)."""
    if freqs[0] > freqs[-1]:
        idx = np.argsort(freqs)
        return power[idx], freqs[idx]
    return power, freqs


def alpha_power_abs(power, freqs):
    """Potencia absoluta en la banda alfa (integración trapezoidal)."""
    p, f = _sort_asc(power, freqs)
    mask = (f >= ALPHA_LO) & (f <= ALPHA_HI)
    if mask.sum() < 2:
        return 0.0
    return float(np.trapezoid(p[mask], f[mask]))


def alpha_power_rel(power, freqs):
    """Potencia relativa: alfa / total (4-40 Hz). Invariante a escala de amplitud."""
    p, f = _sort_asc(power, freqs)
    p_alpha = alpha_power_abs(p, f)
    p_total = float(np.trapezoid(p, f))
    return p_alpha / p_total if p_total > 0 else 0.0


def spectral_entropy(power):
    """
    Entropía espectral de Shannon normalizada.
    0 → energía concentrada en pocas frecuencias (ej. alfa dominante).
    1 → energía distribuida uniformemente.
    """
    total = power.sum()
    if total <= 0:
        return 0.0
    p = power / total
    p = p[p > 0]                        # evitar log(0)
    H     = -np.sum(p * np.log2(p))
    H_max = np.log2(len(power))         # entropía máxima (distribución uniforme)
    return float(H / H_max) if H_max > 0 else 0.0


def alpha_cog(power, freqs):
    """
    Centro de gravedad (CoG) en la banda alfa — método Klimesch (1999).
    Más robusto que el pico máximo cuando hay picos dobles o ambiguos.
    """
    p, f = _sort_asc(power, freqs)
    mask   = (f >= ALPHA_LO) & (f <= ALPHA_HI)
    p_band = p[mask]
    f_band = f[mask]
    denom  = p_band.sum()
    return float(np.sum(f_band * p_band) / denom) if denom > 0 else (ALPHA_LO + ALPHA_HI) / 2.0

## Extracción

### Ventanas

Se heredan las ~232 ventanas de 2 s ya calculadas por la STFT (solapamiento
del 75%, ventana Hann). Para la CWT, se identifica el segmento de muestras
correspondiente a cada ventana STFT y se promedia la potencia en el eje
temporal, obteniendo un espectro representativo de dimensión `(n_freqs,)`.
Esto garantiza que ambas técnicas producen exactamente el mismo número de
instancias con los mismos límites temporales, condición necesaria para que
la comparación sea justa.

### Etiquetado por centro

A cada ventana se le asigna la etiqueta del sample más cercano a su centro
temporal. Esta estrategia es simple, reproducible y suficientemente precisa
para este dataset, donde las transiciones entre estados son lentas relativas
al largo de la ventana (2 s).

### Solapamiento y evaluación posterior

El solapamiento del 75% genera muestras temporalmente dependientes: ventanas
consecutivas comparten 1.5 s de señal. Por esta razón, en el notebook de
clasificación **no se usará un split aleatorio** — se aplicará un split
temporal por bloques (primeros ~80 s para entrenamiento, últimos ~37 s para
test) para evitar data leakage entre conjuntos.

### STFT

In [16]:
stft_freqs  = stft['freqs']     # (n_stft_freqs,)  frecuencias en Hz
stft_times  = stft['times']     # (n_windows,)     tiempo en s de cada ventana
stft_labels = stft['labels']    # (n_windows,)     etiqueta 0/1

FEAT_NAMES = ['alpha_abs', 'alpha_rel', 'entropy', 'cog']
COL_NAMES  = [f'{ch}_{f}' for ch in CHANNELS for f in FEAT_NAMES]

rows = []
for j in range(len(stft_times)):
    row = {}
    for ch in CHANNELS:
        spectrum = stft[ch][:, j]
        row[f'{ch}_alpha_abs'] = alpha_power_abs(spectrum, stft_freqs)
        row[f'{ch}_alpha_rel'] = alpha_power_rel(spectrum, stft_freqs)
        row[f'{ch}_entropy']   = spectral_entropy(spectrum)
        row[f'{ch}_cog']       = alpha_cog(spectrum, stft_freqs)
    rows.append(row)

df_stft_feat = pd.DataFrame(rows, columns=COL_NAMES)
df_stft_feat['label'] = stft_labels.astype(int)

print(f"STFT — shape: {df_stft_feat.shape}")
print(f"  NaN: {df_stft_feat.isnull().sum().sum()}")
print(f"  Clases: {df_stft_feat['label'].value_counts().to_dict()}")

STFT — shape: (236, 57)
  NaN: 0
  Clases: {0: 128, 1: 108}


### CWT

In [17]:
cwt_freqs  = cwt['freqs']
n_cwt_samp = cwt['O2'].shape[1]   # = 14 980

rows = []
for j, t_center in enumerate(stft_times):

    # Índices de muestra CWT correspondientes a esta ventana STFT
    idx_start = max(0,              int(np.round((t_center - WIN_SEC / 2) * FS)))
    idx_end   = min(n_cwt_samp - 1, int(np.round((t_center + WIN_SEC / 2) * FS)))

    row = {}
    for ch in CHANNELS:
        # Promedio temporal dentro de la ventana → espectro (n_freqs,)
        segment  = cwt[ch][:, idx_start : idx_end + 1]
        spectrum = segment.mean(axis=1)

        row[f'{ch}_alpha_abs'] = alpha_power_abs(spectrum, cwt_freqs)
        row[f'{ch}_alpha_rel'] = alpha_power_rel(spectrum, cwt_freqs)
        row[f'{ch}_entropy']   = spectral_entropy(spectrum)
        row[f'{ch}_cog']       = alpha_cog(spectrum, cwt_freqs)
    rows.append(row)

df_cwt_feat = pd.DataFrame(rows, columns=COL_NAMES)
df_cwt_feat['label'] = stft_labels.astype(int)   # mismas ventanas → mismas etiquetas

print(f"CWT — shape: {df_cwt_feat.shape}")
print(f"  NaN: {df_cwt_feat.isnull().sum().sum()}")
print(f"  Clases: {df_cwt_feat['label'].value_counts().to_dict()}")

CWT — shape: (236, 57)
  NaN: 0
  Clases: {0: 128, 1: 108}


### SST

In [18]:
sst_freqs  = sst['freqs']
n_sst_samp = sst['O2'].shape[1]

rows = []
for j, t_center in enumerate(stft_times):

    idx_start = max(0,              int(np.round((t_center - WIN_SEC / 2) * FS)))
    idx_end   = min(n_sst_samp - 1, int(np.round((t_center + WIN_SEC / 2) * FS)))

    row = {}
    for ch in CHANNELS:
        segment  = sst[ch][:, idx_start : idx_end + 1]
        spectrum = segment.mean(axis=1)

        row[f'{ch}_alpha_abs'] = alpha_power_abs(spectrum, sst_freqs)
        row[f'{ch}_alpha_rel'] = alpha_power_rel(spectrum, sst_freqs)
        row[f'{ch}_entropy']   = spectral_entropy(spectrum)
        row[f'{ch}_cog']       = alpha_cog(spectrum, sst_freqs)
    rows.append(row)

df_sst_feat = pd.DataFrame(rows, columns=COL_NAMES)
df_sst_feat['label'] = stft_labels.astype(int)

print(f"SST — shape: {df_sst_feat.shape}")
print(f"  NaN: {df_sst_feat.isnull().sum().sum()}")
print(f"  Clases: {df_sst_feat['label'].value_counts().to_dict()}")

SST — shape: (236, 57)
  NaN: 0
  Clases: {0: 128, 1: 108}


### WPD

#### Funciones especializadas para WPD

Las funciones genéricas (`alpha_power_abs`, `alpha_power_rel`, `alpha_cog`)
usan una máscara binaria de frecuencias, lo cual es apropiado cuando hay
suficiente resolución frecuencial (STFT, CWT, SST). Con la WPD, donde cada
sub-banda tiene 4 Hz de ancho, esa máscara excluye por completo la sub-banda
`[12,16)` aunque el 20% de su ancho (12–13 Hz) pertenece a la banda alfa —
lo que produciría potencia alfa = 0 para todas las ventanas.

La solución es ponderar cada sub-banda por su **fracción de solape** con
8–13 Hz, en lugar de incluirla o excluirla por completo:

$$w_i = \frac{\text{solape}([f_i - 2,\ f_i + 2),\ [8, 13])}{4}$$

donde $f_i$ es la frecuencia central de la sub-banda $i$. Con esta
ponderación, la sub-banda `[8,12)` recibe peso 1.0 (completamente dentro de
alfa) y `[12,16)` recibe peso 0.25 (1 Hz de 4 Hz dentro de alfa) — el mismo
20% de solape cuantificado en el notebook 02.

In [19]:
WPD_BAND_WIDTH = 4.0  # Hz — ancho de cada sub-banda WPD (level=4, fs=128 Hz)

def wpd_band_alpha_weights(freqs, band_width=WPD_BAND_WIDTH):
    """
    Calcula la fracción de cada sub-banda WPD que cae dentro de la banda
    alfa (8-13 Hz). Cada sub-banda i tiene centro freqs[i] y bordes
    [freqs[i] - band_width/2, freqs[i] + band_width/2).
    """
    lo_edges = freqs - band_width / 2
    hi_edges = freqs + band_width / 2
    overlap  = np.clip(np.minimum(hi_edges, ALPHA_HI)
                       - np.maximum(lo_edges, ALPHA_LO), 0, None)
    return overlap / band_width


def wpd_alpha_power_abs(power, freqs, band_width=WPD_BAND_WIDTH):
    """Potencia absoluta alfa para WPD, ponderada por fracción de solape."""
    weights = wpd_band_alpha_weights(freqs, band_width)
    return float(np.sum(power * weights * band_width))


def wpd_alpha_power_rel(power, freqs, band_width=WPD_BAND_WIDTH):
    """Potencia relativa alfa para WPD: alfa ponderada / total."""
    p_alpha = wpd_alpha_power_abs(power, freqs, band_width)
    p_total = float(np.sum(power) * band_width)
    return p_alpha / p_total if p_total > 0 else 0.0


def wpd_alpha_cog(power, freqs, band_width=WPD_BAND_WIDTH):
    """Centro de gravedad alfa para WPD, ponderado por fracción de solape."""
    weights = wpd_band_alpha_weights(freqs, band_width)
    p_band  = power * weights
    denom   = p_band.sum()
    return float(np.sum(freqs * p_band) / denom) if denom > 0 else (ALPHA_LO + ALPHA_HI) / 2.0


# Nota: spectral_entropy no necesita versión especializada — opera sobre
# el espectro completo de 9 sub-bandas sin distinguir la banda alfa.

In [20]:
wpd_freqs = wpd['freqs']
wpd_times = wpd['times']

# Para cada ventana STFT, encontrar la ventana WPD con tiempo central
# más cercano (ambas usan window_sec=2 y overlap=0.75, pero el manejo
# de bordes difiere entre scipy.signal.stft y el ventaneo manual de WPD)
wpd_idx_map = np.array([
    np.argmin(np.abs(wpd_times - ti)) for ti in stft_times
])

max_offset_ms = np.max(np.abs(wpd_times[wpd_idx_map] - stft_times)) * 1000
print(f"Offset temporal máximo STFT-WPD: {max_offset_ms:.1f} ms "
      f"(ventana = {WIN_SEC*1000:.0f} ms)")

rows = []
for j, idx in enumerate(wpd_idx_map):
    row = {}
    for ch in CHANNELS:
        spectrum = wpd[ch][:, idx]   # ya es potencia promedio de esa ventana

        row[f'{ch}_alpha_abs'] = wpd_alpha_power_abs(spectrum, wpd_freqs)
        row[f'{ch}_alpha_rel'] = wpd_alpha_power_rel(spectrum, wpd_freqs)
        row[f'{ch}_entropy']   = spectral_entropy(spectrum)
        row[f'{ch}_cog']       = wpd_alpha_cog(spectrum, wpd_freqs)
    rows.append(row)

df_wpd_feat = pd.DataFrame(rows, columns=COL_NAMES)
df_wpd_feat['label'] = stft_labels.astype(int)

print(f"WPD — shape: {df_wpd_feat.shape}")
print(f"  NaN: {df_wpd_feat.isnull().sum().sum()}")
print(f"  Clases: {df_wpd_feat['label'].value_counts().to_dict()}")

Offset temporal máximo STFT-WPD: 1500.0 ms (ventana = 2000 ms)
WPD — shape: (236, 57)
  NaN: 0
  Clases: {0: 128, 1: 108}


In [21]:
offsets = np.abs(wpd_times[wpd_idx_map] - stft_times)

print("Distribución del offset temporal STFT-WPD (segundos):")
print(f"  Mínimo:     {offsets.min():.4f}")
print(f"  Percentil 50: {np.percentile(offsets, 50):.4f}")
print(f"  Percentil 90: {np.percentile(offsets, 90):.4f}")
print(f"  Percentil 99: {np.percentile(offsets, 99):.4f}")
print(f"  Máximo:     {offsets.max():.4f}")

print(f"\nVentanas con offset > 0.1s: {(offsets > 0.1).sum()} de {len(offsets)}")
print(f"Índices de esas ventanas: {np.where(offsets > 0.1)[0]}")
print(f"Tiempos STFT de esas ventanas: {stft_times[offsets > 0.1]}")

Distribución del offset temporal STFT-WPD (segundos):
  Mínimo:     0.0000
  Percentil 50: 0.0000
  Percentil 90: 0.0000
  Percentil 99: 0.8250
  Máximo:     1.5000

Ventanas con offset > 0.1s: 5 de 236
Índices de esas ventanas: [  0   1 233 234 235]
Tiempos STFT de esas ventanas: [  0.    0.5 116.5 117.  117.5]


### Exclusión de ventanas de borde por desalineación temporal

Al integrar la WPD (calculada con ventaneo manual) con la grilla temporal de
la STFT (calculada con `scipy.signal.stft`), se detectó que 5 de las 236
ventanas — las 2 primeras y las 3 últimas del registro — presentan offset
temporal significativo (hasta 1.5 s) entre ambas técnicas. Esto se debe a
una diferencia de convención: `scipy.signal.stft` aplica zero-padding en los
bordes por defecto, mientras que el ventaneo manual de WPD no lo hace.

Para garantizar que las cuatro técnicas (STFT, CWT, SST, WPD) se comparan
exactamente sobre los mismos segmentos temporales, se excluyen estas 5
ventanas de borde de todos los datasets de features, dejando 231 instancias
válidas. El 95.8% de las ventanas originales mantiene offset = 0.0 (alineación
perfecta), por lo que esta exclusión tiene impacto marginal en el tamaño
muestral.

In [22]:
# Excluir ventanas de borde donde STFT y WPD no están temporalmente alineadas
# (ver diagnóstico: 5 de 236 ventanas con offset > 0.1s, todas en los extremos
# del registro, debido a la diferencia de convención de padding entre
# scipy.signal.stft (zero-padding en bordes) y el ventaneo manual de WPD)
OFFSET_THRESHOLD = 0.1  # segundos
valid_mask = offsets <= OFFSET_THRESHOLD

print(f"Ventanas excluidas por desalineación de borde: {(~valid_mask).sum()} de {len(valid_mask)}")
print(f"Ventanas WPD válidas: {valid_mask.sum()}")

Ventanas excluidas por desalineación de borde: 5 de 236
Ventanas WPD válidas: 231


In [23]:
df_wpd_feat = df_wpd_feat.loc[valid_mask].reset_index(drop=True)
print(f"\nWPD — shape tras filtrar bordes: {df_wpd_feat.shape}")


WPD — shape tras filtrar bordes: (231, 57)


In [24]:
df_stft_feat = df_stft_feat.loc[valid_mask].reset_index(drop=True)
df_cwt_feat  = df_cwt_feat.loc[valid_mask].reset_index(drop=True)
df_sst_feat  = df_sst_feat.loc[valid_mask].reset_index(drop=True)

print(f"Shape final (las 4 técnicas alineadas): {df_stft_feat.shape}")

Shape final (las 4 técnicas alineadas): (231, 57)


## Revisión

In [30]:
print("=== Verificación del pipeline ===")
print(f"STFT freqs: {stft_freqs.min():.2f} - {stft_freqs.max():.2f} Hz  ({len(stft_freqs)} bins)")
print(f"CWT  freqs: {cwt_freqs.min():.2f} - {cwt_freqs.max():.2f} Hz  ({len(cwt_freqs)} bins)")
print(f"SST  freqs: {sst_freqs.min():.2f} - {sst_freqs.max():.2f} Hz  ({len(sst_freqs)} bins)")
print(f"WPD  freqs: {wpd_freqs.min():.2f} - {wpd_freqs.max():.2f} Hz  ({len(wpd_freqs)} bins)")

# ¿La señal tiene media ~0 como corresponde tras Z-score?
import pandas as pd
df_orig = pd.read_csv('./data/eeg_eye_state_clean.csv')
print(f"\nMedia O1 en señal preprocesada: {df_orig['O1'].mean():.4f}  (esperado: ~0)")
print(f"Std  O1 en señal preprocesada: {df_orig['O1'].std():.4f}   (esperado: ~1)")

# Distribución temporal de etiquetas
labels_all = df_orig['eyeDetection'].values
print(f"\nTotal muestras clase 0: {(labels_all==0).sum()}  ({100*(labels_all==0).mean():.1f}%)")
print(f"Total muestras clase 1: {(labels_all==1).sum()}  ({100*(labels_all==1).mean():.1f}%)")

# Verificar consistencia entre las 4 técnicas
print(f"\nShapes finales: STFT={df_stft_feat.shape}, CWT={df_cwt_feat.shape}, "
      f"SST={df_sst_feat.shape}, WPD={df_wpd_feat.shape}")
labels_iguales = (df_stft_feat['label'].values == df_cwt_feat['label'].values).all() and \
                 (df_stft_feat['label'].values == df_sst_feat['label'].values).all() and \
                 (df_stft_feat['label'].values == df_wpd_feat['label'].values).all()
print(f"¿Etiquetas idénticas entre las 4 técnicas? {'Si' if labels_iguales else 'No'}")

=== Verificación del pipeline ===
STFT freqs: 4.00 - 40.00 Hz  (73 bins)
CWT  freqs: 4.00 - 40.00 Hz  (80 bins)
SST  freqs: 4.00 - 40.00 Hz  (80 bins)
WPD  freqs: 6.00 - 38.00 Hz  (9 bins)

Media O1 en señal preprocesada: 0.0000  (esperado: ~0)
Std  O1 en señal preprocesada: 1.0000   (esperado: ~1)

Total muestras clase 0: 8257  (55.1%)
Total muestras clase 1: 6723  (44.9%)

Shapes finales: STFT=(231, 57), CWT=(231, 57), SST=(231, 57), WPD=(231, 57)
¿Etiquetas idénticas entre las 4 técnicas? Si


## Guardado

In [31]:
output_dir = './data/features/'

os.makedirs(output_dir, exist_ok=True)

df_stft_feat.to_csv(output_dir + 'features_stft.csv', index=False)
df_cwt_feat.to_csv(output_dir + 'features_cwt.csv',  index=False)
df_sst_feat.to_csv(output_dir + 'features_sst.csv',  index=False)
df_wpd_feat.to_csv(output_dir + 'features_wpd.csv',  index=False)

print(f"Guardado: features_stft.csv  — {df_stft_feat.shape}")
print(f"Guardado: features_cwt.csv   — {df_cwt_feat.shape}")
print(f"Guardado: features_sst.csv   — {df_sst_feat.shape}")
print(f"Guardado: features_wpd.csv   — {df_wpd_feat.shape}")
print(f"\nColumnas de features: {COL_NAMES[:8]} ... ({len(COL_NAMES)} total + label)")

Guardado: features_stft.csv  — (231, 57)
Guardado: features_cwt.csv   — (231, 57)
Guardado: features_sst.csv   — (231, 57)
Guardado: features_wpd.csv   — (231, 57)

Columnas de features: ['AF3_alpha_abs', 'AF3_alpha_rel', 'AF3_entropy', 'AF3_cog', 'F7_alpha_abs', 'F7_alpha_rel', 'F7_entropy', 'F7_cog'] ... (56 total + label)
